In [2]:
import json
import time
import psutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import f_oneway
from mealpy import FloatVar
from mealpy.swarm_based import PSO, GWO, HHO
from mealpy.evolutionary_based import DE, GA

def load_and_prepare_data():
    url = "https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv"
    df = pd.read_csv(url)
    X = df.drop('charges', axis=1)
    y = df['charges'].values.reshape(-1, 1)
    categorical_cols = ['sex', 'smoker', 'region']
    numerical_cols = ['age', 'bmi', 'children']
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_cols),
            ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
        ])
    X_processed = preprocessor.fit_transform(X)
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y)
    return X_processed, y_scaled, scaler_y, df

def check_heterogeneity(clients_data):
    groups = [client['y'] for client in clients_data]
    f_stat, p_value = f_oneway(*groups)
    f_stat = float(f_stat)
    p_value = float(p_value)
    print(f"ANOVA test for heterogeneity: F={f_stat:.4f}, p={p_value:.4e}")
    print("Clients are significantly heterogeneous: " + ("True" if p_value < 0.05 else "False"))

def split_into_clients(df, n_clients=5):
    df_grouped = df.groupby(['region', 'smoker'])
    client_dfs = [group for _, group in df_grouped]
    while len(client_dfs) < n_clients:
        largest = max(client_dfs, key=len)
        client_dfs.remove(largest)
        split_idx = len(largest) // 2
        client_dfs.append(largest.iloc[:split_idx])
        client_dfs.append(largest.iloc[split_idx:])
    clients = []
    for cdf in client_dfs[:n_clients]:
        X = cdf.drop('charges', axis=1)
        y = cdf['charges'].values.reshape(-1, 1)
        clients.append({'X': X, 'y': y})
    check_heterogeneity(clients)
    return clients

class RegressionNet(nn.Module):
    def __init__(self, input_dim):
        super(RegressionNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

def train_local_model(model, X_train, y_train, device, epochs=10, lr=0.001):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    for _ in range(epochs):
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

def evaluate_model(model, X_test, y_test, device):
    model.eval()
    with torch.no_grad():
        X_test_t = torch.FloatTensor(X_test).to(device)
        preds = model(X_test_t).cpu().numpy().flatten()
        y_test_flat = y_test.flatten()
    rmse = np.sqrt(mean_squared_error(y_test_flat, preds))
    mae = mean_absolute_error(y_test_flat, preds)
    r2 = r2_score(y_test_flat, preds)
    return rmse, mae, r2

def fedprox_aggregate(global_model, client_models, mu, device):
    global_dict = global_model.state_dict()
    for key in global_dict.keys():
        if 'weight' in key or 'bias' in key:
            other_params = [cm.state_dict()[key].float() for cm in client_models]
            avg = torch.mean(torch.stack(other_params), dim=0)
            proximal_diff = mu * (global_dict[key] - avg)
            global_dict[key] = avg + proximal_diff
    global_model.load_state_dict(global_dict)

def get_method_optimizer(method_name, problem_dict):
    if method_name == "DE":
        return DE.OriginalDE(epoch=100, pop_size=50, **problem_dict)
    elif method_name == "PSO":
        return PSO.OriginalPSO(epoch=100, pop_size=50, **problem_dict)
    elif method_name == "GWO":
        return GWO.OriginalGWO(epoch=100, pop_size=50, **problem_dict)
    elif method_name == "HHO":
        return HHO.OriginalHHO(epoch=100, pop_size=50, **problem_dict)
    elif method_name == "GA":
        return GA.OriginalGA(epoch=100, pop_size=50, **problem_dict)
    return None

def tune_proximal_param(method_name, client_models, global_model, device, input_dim, clients):
    if not client_models or len(client_models) != len(clients):
        raise ValueError("Client models list mismatch or empty")

    start_time = time.time()
    process = psutil.Process()
    start_mem = process.memory_info().rss / (1024 ** 2)

    def objective(mu_vec):
        mu = mu_vec[0]
        temp_global = RegressionNet(input_dim).to(device)
        temp_global.load_state_dict(global_model.state_dict())
        fedprox_aggregate(temp_global, client_models, mu, device)
        total_loss = 0.0
        for i in range(len(clients)):
            _, _, r2 = evaluate_model(temp_global, clients[i]['X_test'], clients[i]['y_test'], device)
            total_loss += (1 - r2)
        return total_loss / len(client_models)

    problem_dict = {
        "obj_func": objective,
        "bounds": FloatVar(lb=[0.001], ub=[1.0]),
        "minmax": "min",
        "verbose": False
    }

    if method_name in ["Grid Search", "Random Search"]:
        candidates = np.linspace(0.001, 1.0, 40)
        if method_name == "Random Search":
            np.random.shuffle(candidates)
        best_mu = 0.001
        best_score = float('inf')
        for mu in candidates[:30]:
            score = objective([mu])
            if score < best_score:
                best_score = score
                best_mu = mu
        mu_best = best_mu
    else:
        optimizer = get_method_optimizer(method_name, problem_dict)
        if optimizer is None:
            raise ValueError(f"Unsupported method: {method_name}")
        best_agent = optimizer.solve(problem=problem_dict)
        mu_best = best_agent.solution[0]

    end_time = time.time()
    end_mem = process.memory_info().rss / (1024 ** 2)
    search_time = end_time - start_time
    total_memory = max(0.0, end_mem - start_mem)

    return mu_best, search_time, total_memory

def run_experiment():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    _, _, _, df_raw = load_and_prepare_data()
    clients_raw = split_into_clients(df_raw)

    clients = []
    for c in clients_raw:
        X_c = c['X']
        y_c = c['y']
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', StandardScaler(), ['age', 'bmi', 'children']),
                ('cat', OneHotEncoder(drop='first', sparse_output=False), ['sex', 'smoker', 'region'])
            ])
        X_c_processed = preprocessor.fit_transform(X_c)
        scaler_y_c = StandardScaler()
        y_c_scaled = scaler_y_c.fit_transform(y_c)
        clients.append({'X': X_c_processed, 'y': y_c_scaled, 'preprocessor': preprocessor, 'scaler_y': scaler_y_c})

    input_dim = clients[0]['X'].shape[1]
    methods = ["DE", "PSO", "GWO", "HHO", "GA", "Grid Search", "Random Search"]

    results = {m: {} for m in methods}

    for fold in range(1, 11):
        print(f"Starting Fold {fold}")
        fold_data = {m: {'local': [], 'global': []} for m in methods}

        for round_num in range(1, 51):
            print(f"  Round {round_num}")
            kf = KFold(n_splits=10, shuffle=True, random_state=fold * 42 + round_num)

            global_models = {m: RegressionNet(input_dim).to(device) for m in methods}

            local_rmse_list = {m: [] for m in methods}
            local_mae_list = {m: [] for m in methods}
            local_r2_list = {m: [] for m in methods}

            client_models_per_method = {m: [] for m in methods}

            for client_idx, client in enumerate(clients):
                splits = list(kf.split(client['X']))
                train_idx, test_idx = splits[0]  # Gunakan split pertama setiap round (atau bisa random index jika ingin variasi lebih)
                X_train, X_test = client['X'][train_idx], client['X'][test_idx]
                y_train, y_test = client['y'][train_idx], client['y'][test_idx]
                client['X_test'] = X_test
                client['y_test'] = y_test

                for m in methods:
                    model = RegressionNet(input_dim).to(device)
                    if round_num > 1:
                        model.load_state_dict(global_models[m].state_dict())
                    train_local_model(model, X_train, y_train, device)
                    rmse, mae, r2 = evaluate_model(model, X_test, y_test, device)
                    local_rmse_list[m].append(rmse)
                    local_mae_list[m].append(mae)
                    local_r2_list[m].append(r2)
                    client_models_per_method[m].append(model)

            for m in methods:
                avg_local_rmse = np.mean(local_rmse_list[m])
                avg_local_mae = np.mean(local_mae_list[m])
                avg_local_r2 = np.mean(local_r2_list[m])

                fold_data[m]['local'].append({
                    'rmse': float(avg_local_rmse),
                    'mae': float(avg_local_mae),
                    'r2': float(avg_local_r2)
                })

                mu, search_time, total_mem = tune_proximal_param(
                    m, client_models_per_method[m], global_models[m], device, input_dim, clients
                )
                fedprox_aggregate(global_models[m], client_models_per_method[m], mu, device)

                global_rmse_list = []
                global_mae_list = []
                global_r2_list = []
                for client in clients:
                    rmse_g, mae_g, r2_g = evaluate_model(global_models[m], client['X_test'], client['y_test'], device)
                    global_rmse_list.append(rmse_g)
                    global_mae_list.append(mae_g)
                    global_r2_list.append(r2_g)

                avg_global_rmse = np.mean(global_rmse_list)
                avg_global_mae = np.mean(global_mae_list)
                avg_global_r2 = np.mean(global_r2_list)

                fold_data[m]['global'].append({
                    'mu': float(mu),
                    'search_time': float(search_time),
                    'total_memory': float(total_mem),
                    'rmse': float(avg_global_rmse),
                    'mae': float(avg_global_mae),
                    'r2': float(avg_global_r2)
                })

                print(f"fold {fold} round {round_num} ({m})")
                print(f"local= RMSE: {avg_local_rmse:.4f} | MAE: {avg_local_mae:.4f} | R2: {avg_local_r2:.4f}")
                print(f"global= nilai proximal: {mu:.4f} | waktu pencarian: {search_time:.4f} | memory yang digunakan: {total_mem:.4f} | RMSE: {avg_global_rmse:.4f} | MAE: {avg_global_mae:.4f} | R2: {avg_global_r2:.4f}")

        results_fold = {}
        for m in methods:
            local_rmse = [d['rmse'] for d in fold_data[m]['local']]
            local_mae = [d['mae'] for d in fold_data[m]['local']]
            local_r2 = [d['r2'] for d in fold_data[m]['local']]
            global_mu = [d['mu'] for d in fold_data[m]['global']]
            global_time = [d['search_time'] for d in fold_data[m]['global']]
            global_mem = [d['total_memory'] for d in fold_data[m]['global']]
            global_rmse = [d['rmse'] for d in fold_data[m]['global']]
            global_mae = [d['mae'] for d in fold_data[m]['global']]
            global_r2 = [d['r2'] for d in fold_data[m]['global']]

            results_fold[m] = {
                'local_rmse': local_rmse,
                'local_mae': local_mae,
                'local_r2': local_r2,
                'global_mu': global_mu,
                'global_search_time': global_time,
                'global_total_memory': global_mem,
                'global_rmse': global_rmse,
                'global_mae': global_mae,
                'global_r2': global_r2
            }

            print(f"fold average round {round_num} ({m})")
            print(f"lokal= RMSE: {np.mean(local_rmse):.4f} ± {np.std(local_rmse):.4f} | MAE: {np.mean(local_mae):.4f} ± {np.std(local_mae):.4f} | R2: {np.mean(local_r2):.4f} ± {np.std(local_r2):.4f}")
            print(f"global= nilai proximal: {np.mean(global_mu):.4f} ± {np.std(global_mu):.4f} | waktu pencarian: {np.mean(global_time):.4f} ± {np.std(global_time):.4f} | memory yang digunakan: {np.mean(global_mem):.4f} ± {np.std(global_mem):.4f} | RMSE: {np.mean(global_rmse):.4f} ± {np.std(global_rmse):.4f} | MAE: {np.mean(global_mae):.4f} ± {np.std(global_mae):.4f} | R2: {np.mean(global_r2):.4f} ± {np.std(global_r2):.4f}")

        results[f"fold_{fold}"] = results_fold

    with open('fedprox_results.json', 'w') as f:
        json.dump(results, f, indent=4)
    print("All results saved to fedprox_results.json")

if __name__ == "__main__":
    run_experiment()

C:\Users\Michael Angello\AppData\Local\Temp\ipykernel_15276\2680672437.py:39: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  f_stat = float(f_stat)
C:\Users\Michael Angello\AppData\Local\Temp\ipykernel_15276\2680672437.py:40: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  p_value = float(p_value)


ANOVA test for heterogeneity: F=244.9053, p=4.6609e-143
Clients are significantly heterogeneous: True
Starting Fold 1
  Round 1


2026/01/21 10:38:35 PM, INFO, mealpy.evolutionary_based.DE.OriginalDE: OriginalDE(epoch=100, pop_size=50, wf=0.1, cr=0.9, strategy=0)
2026/01/21 10:38:36 PM, INFO, mealpy.evolutionary_based.DE.OriginalDE: >>>Problem: P, Epoch: 1, Current best: 1.1599377416755523, Global best: 1.1599377416755523, Runtime: 0.50060 seconds
2026/01/21 10:38:37 PM, INFO, mealpy.evolutionary_based.DE.OriginalDE: >>>Problem: P, Epoch: 2, Current best: 1.159516110979824, Global best: 1.159516110979824, Runtime: 0.48107 seconds
2026/01/21 10:38:37 PM, INFO, mealpy.evolutionary_based.DE.OriginalDE: >>>Problem: P, Epoch: 3, Current best: 1.159516110979824, Global best: 1.159516110979824, Runtime: 0.50470 seconds
2026/01/21 10:38:38 PM, INFO, mealpy.evolutionary_based.DE.OriginalDE: >>>Problem: P, Epoch: 4, Current best: 1.159516110979824, Global best: 1.159516110979824, Runtime: 0.49280 seconds
2026/01/21 10:38:38 PM, INFO, mealpy.evolutionary_based.DE.OriginalDE: >>>Problem: P, Epoch: 5, Current best: 1.15951611

fold 1 round 1 (DE)
local= RMSE: 0.7556 | MAE: 0.6120 | R2: 0.3558
global= nilai proximal: 0.0010 | waktu pencarian: 49.9733 | memory yang digunakan: 0.0000 | RMSE: 0.9969 | MAE: 0.8413 | R2: -0.1595


2026/01/21 10:39:26 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 1, Current best: 1.1208499082895391, Global best: 1.1208499082895391, Runtime: 0.51263 seconds
2026/01/21 10:39:27 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 2, Current best: 1.1203431322351332, Global best: 1.1203431322351332, Runtime: 0.50437 seconds
2026/01/21 10:39:27 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 3, Current best: 1.1203431322351332, Global best: 1.1203431322351332, Runtime: 0.50896 seconds
2026/01/21 10:39:28 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 4, Current best: 1.1203431322351332, Global best: 1.1203431322351332, Runtime: 0.49308 seconds
2026/01/21 10:39:28 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoch: 5, Current best: 1.1203431322351332, Global best: 1.1203431322351332, Runtime: 0.49909 seconds
2026/01/21 10:39:29 PM, INFO, mealpy.swarm_based.PSO.OriginalPSO: >>>Problem: P, Epoc

fold 1 round 1 (PSO)
local= RMSE: 0.7125 | MAE: 0.5657 | R2: 0.4111
global= nilai proximal: 1.0000 | waktu pencarian: 50.4145 | memory yang digunakan: 0.0000 | RMSE: 0.9900 | MAE: 0.8347 | R2: -0.1203


... (4,992 outputs left out)

2026/01/23 12:08:36 PM, INFO, mealpy.swarm_based.HHO.OriginalHHO: >>>Problem: P, Epoch: 1, Current best: 1.1894646326496567, Global best: 1.1894646326496567, Runtime: 0.83741 seconds
2026/01/23 12:08:37 PM, INFO, mealpy.swarm_based.HHO.OriginalHHO: >>>Problem: P, Epoch: 2, Current best: 1.1894370603600342, Global best: 1.1894370603600342, Runtime: 0.80293 seconds
2026/01/23 12:08:38 PM, INFO, mealpy.swarm_based.HHO.OriginalHHO: >>>Problem: P, Epoch: 3, Current best: 1.1894352502998697, Global best: 1.1894352502998697, Runtime: 0.80049 seconds
2026/01/23 12:08:38 PM, INFO, mealpy.swarm_based.HHO.OriginalHHO: >>>Problem: P, Epoch: 4, Current best: 1.1894352502998697, Global best: 1.1894352502998697, Runtime: 0.84527 seconds
2026/01/23 12:08:39 PM, INFO, mealpy.swarm_based.HHO.OriginalHHO: >>>Problem: P, Epoch: 5, Current best: 1.1894352502998697, Global best: 1.1894352502998697, Runtime: 0.76574 seconds
2026/01/23 12:08:40 PM, INFO, mealpy.swarm_based.HHO.OriginalHHO: >>>Problem: P, Epoc

fold 10 round 50 (HHO)
local= RMSE: 0.8193 | MAE: 0.5481 | R2: 0.0048
global= nilai proximal: 0.2862 | waktu pencarian: 98.3743 | memory yang digunakan: 0.0000 | RMSE: 0.8838 | MAE: 0.6873 | R2: -0.1894


2026/01/23 12:10:13 PM, INFO, mealpy.evolutionary_based.GA.OriginalGA: >>>Problem: P, Epoch: 1, Current best: 1.2849868215846176, Global best: 1.2849868215846176, Runtime: 0.00008 seconds
2026/01/23 12:10:13 PM, INFO, mealpy.evolutionary_based.GA.OriginalGA: >>>Problem: P, Epoch: 2, Current best: 1.2849868215846176, Global best: 1.2849868215846176, Runtime: 0.00018 seconds
2026/01/23 12:10:13 PM, INFO, mealpy.evolutionary_based.GA.OriginalGA: >>>Problem: P, Epoch: 3, Current best: 1.2849868215846176, Global best: 1.2849868215846176, Runtime: 0.00011 seconds
2026/01/23 12:10:13 PM, INFO, mealpy.evolutionary_based.GA.OriginalGA: >>>Problem: P, Epoch: 4, Current best: 1.2849868215846176, Global best: 1.2849868215846176, Runtime: 0.00041 seconds
2026/01/23 12:10:13 PM, INFO, mealpy.evolutionary_based.GA.OriginalGA: >>>Problem: P, Epoch: 5, Current best: 1.2849868215846176, Global best: 1.2849868215846176, Runtime: 0.00014 seconds
2026/01/23 12:10:13 PM, INFO, mealpy.evolutionary_based.GA.O

fold 10 round 50 (GA)
local= RMSE: 0.8638 | MAE: 0.6125 | R2: -0.1200
global= nilai proximal: 0.5512 | waktu pencarian: 0.7342 | memory yang digunakan: 0.3008 | RMSE: 0.9078 | MAE: 0.7258 | R2: -0.2850
fold 10 round 50 (Grid Search)
local= RMSE: 0.8605 | MAE: 0.5928 | R2: -0.0980
global= nilai proximal: 0.4877 | waktu pencarian: 0.3482 | memory yang digunakan: 0.0000 | RMSE: 0.8824 | MAE: 0.7111 | R2: -0.1728
fold 10 round 50 (Random Search)
local= RMSE: 0.9136 | MAE: 0.6472 | R2: -0.2829
global= nilai proximal: 0.2315 | waktu pencarian: 0.3678 | memory yang digunakan: 0.0234 | RMSE: 0.9295 | MAE: 0.7158 | R2: -0.3636
fold average round 50 (DE)
lokal= RMSE: 0.7527 ± 0.0843 | MAE: 0.5468 ± 0.0572 | R2: 0.3125 ± 0.1974
global= nilai proximal: 0.0248 ± 0.0777 | waktu pencarian: 56.5101 ± 6.0461 | memory yang digunakan: 0.0771 ± 0.1716 | RMSE: 0.7965 ± 0.0859 | MAE: 0.6155 ± 0.0592 | R2: 0.2314 ± 0.1947
fold average round 50 (PSO)
lokal= RMSE: 0.7596 ± 0.0800 | MAE: 0.5525 ± 0.0492 | R2: 0